# OCTA-500 Capillary: Single U-Net Training Run

Trains one plain U-Net on OCTA-500's capillary ground truth (`GT_Capillary`,
`ILM_OPL` projection, `dataset.get_octa500_split()`'s 40-image held-out test
set), then loads fold 1's checkpoint and runs it on a few subjects as a
sanity check. This is the single-variant precursor to the architecture
ablation in `scripts/octa500_lv_ablation.py` — that script sweeps all 5
FlexUNet variants but only on the 6mm large-vessel GT; this notebook is the
only place capillary GT gets a trained model.


In [3]:
import sys
sys.path.append('/users/egottfri/code/octa-segmentation/src')
from train import run_kfold
from pathlib import Path
from sklearn.model_selection import train_test_split
print("imports done")

imports done


In [4]:
OCTA500 = Path("/files22_lrsresearch/ENG_Lee-Lab_Shared/group/data/public/OCTA_500")
PROJ    = OCTA500 / "OCTA_3mm/Projection Maps/OCTA(ILM_OPL)"
LABELS  = OCTA500 / "Label/Label/GT_Capillary"

ids = [str(i) for i in range(10301, 10501)]
all_imgs  = [PROJ   / f"{i}.bmp" for i in ids]
all_masks = [LABELS / f"{i}.bmp" for i in ids]

train_imgs, test_imgs, train_masks, test_masks = train_test_split(
    all_imgs, all_masks, test_size=40, random_state=42
)
print(f"Train: {len(train_imgs)}, Test: {len(test_imgs)}")

# Verify a few files actually exist
for p in all_imgs[:3]:
    print(p, "exists:", p.exists())

Train: 160, Test: 40
/files22_lrsresearch/ENG_Lee-Lab_Shared/group/data/public/OCTA_500/OCTA_3mm/Projection Maps/OCTA(ILM_OPL)/10301.bmp exists: True
/files22_lrsresearch/ENG_Lee-Lab_Shared/group/data/public/OCTA_500/OCTA_3mm/Projection Maps/OCTA(ILM_OPL)/10302.bmp exists: True
/files22_lrsresearch/ENG_Lee-Lab_Shared/group/data/public/OCTA_500/OCTA_3mm/Projection Maps/OCTA(ILM_OPL)/10303.bmp exists: True


In [6]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

scores = run_kfold(
    train_imgs, train_masks, test_imgs, test_masks,
    n_splits=5, num_epochs=100, patience=10, batch_size=4,
    lr=1e-4, device=device,
    output_dir='results/octa500_capillary'
)

device: cuda

FOLD 1/5
Epoch 1/100 | train_loss: 0.3974 | val_loss: 0.4878 | val_dice: 0.4912
New best! Saved (val_dice: 0.4912)
Epoch 2/100 | train_loss: 0.2825 | val_loss: 0.2687 | val_dice: 0.8728
New best! Saved (val_dice: 0.8728)
Epoch 3/100 | train_loss: 0.2564 | val_loss: 0.2470 | val_dice: 0.8817
New best! Saved (val_dice: 0.8817)
Epoch 4/100 | train_loss: 0.2398 | val_loss: 0.2340 | val_dice: 0.8851
New best! Saved (val_dice: 0.8851)
Epoch 5/100 | train_loss: 0.2274 | val_loss: 0.2270 | val_dice: 0.8781
Epoch 6/100 | train_loss: 0.2190 | val_loss: 0.2144 | val_dice: 0.8859
New best! Saved (val_dice: 0.8859)
Epoch 7/100 | train_loss: 0.2131 | val_loss: 0.2090 | val_dice: 0.8886
New best! Saved (val_dice: 0.8886)
Epoch 8/100 | train_loss: 0.2073 | val_loss: 0.2045 | val_dice: 0.8894
New best! Saved (val_dice: 0.8894)
Epoch 9/100 | train_loss: 0.2020 | val_loss: 0.2018 | val_dice: 0.8854
Epoch 10/100 | train_loss: 0.1996 | val_loss: 0.2056 | val_dice: 0.8854
Epoch 11/100 | train_

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from model import build_model, remap_legacy_state_dict

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load fold 1's best model (may already be saved even mid-training)
model = build_model('unet', in_channels=1, num_classes=1).to(device)
state_dict = torch.load(
    'results/octa500_capillary/best_model_fold1.pth',
    map_location=device
)
model.load_state_dict(remap_legacy_state_dict(state_dict))
model.eval()

# Pick a few OCTA-500 images to run through it
OCTA500 = Path("/files22_lrsresearch/ENG_Lee-Lab_Shared/group/data/public/OCTA_500")
PROJ    = OCTA500 / "OCTA_3mm/Projection Maps/OCTA(ILM_OPL)"
LABELS  = OCTA500 / "Label/Label/GT_Capillary"

subject_ids = ["10301", "10302", "10303"]

fig, axes = plt.subplots(len(subject_ids), 3, figsize=(12, 4 * len(subject_ids)))

for row, sid in enumerate(subject_ids):
    # Load image and ground truth
    img  = cv2.imread(str(PROJ   / f"{sid}.bmp"), cv2.IMREAD_GRAYSCALE)
    gt   = cv2.imread(str(LABELS / f"{sid}.bmp"), cv2.IMREAD_GRAYSCALE)

    # Preprocess exactly as training did: normalize to [0,1], add batch+channel dims
    img_tensor = torch.tensor(img / 255.0, dtype=torch.float32)
    img_tensor = img_tensor.unsqueeze(0).unsqueeze(0).to(device)  # (1,1,H,W)

    with torch.no_grad():
        logits = model(img_tensor)
        pred = (torch.sigmoid(logits) > 0.5).float()
        pred_np = pred.cpu().numpy().squeeze()  # (H,W)

    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title(f"Subject {sid} — raw image")
    axes[row, 0].axis('off')

    axes[row, 1].imshow(gt, cmap='gray')
    axes[row, 1].set_title("Ground truth")
    axes[row, 1].axis('off')

    axes[row, 2].imshow(pred_np, cmap='gray')
    axes[row, 2].set_title("U-Net prediction")
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig('results/octa500_capillary/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("saved.")

saved.
